In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

PASTA_DADOS = Path('dados_tratados')
META_ANUAL = 88000
META_COLABORATIVA = 70000

In [8]:
def carregar_dados_tratados(caminho=PASTA_DADOS / 'projetos_tratados.csv'):
    return pd.read_csv(caminho, parse_dates=['data_inicio', 'data_fim', 'data_registro'])

df = carregar_dados_tratados()
df.head()

,data_registro,nome_projeto,area,responsavel,cliente,satisfacao,valor_total_projeto,valor_faturamento_colaborativo,projeto_de_impacto,data_inicio,data_fim,status,ano_inicio,mes_inicio,ano_mes_inicio,duracao_dias,valor_faturamento_proprio
0,2026-09-07 15:31:26,Projeto Umbra,Mobile,Juliana Lima,Aurora Labs,5.0,2500,0,False,2024-03-09,2024-04-09,Concluído,2024,3,2024-03,31,2500
1,2026-09-06 15:31:26,Projeto Constelação,Sites,Marcos Vinícius,Delta Consult,5.0,5000,5000,False,2024-04-19,2024-05-19,Concluído,2024,4,2024-04,30,0
2,2026-09-05 15:31:26,Projeto Solstício,Mobile,Rafael Souza,NimbusSoft,4.0,3000,0,False,2024-05-11,2024-06-11,Concluído,2024,5,2024-05,31,3000
3,2026-09-04 15:31:26,Projeto Vetor,Desktop,Pedro de Pádua,Jagger CO,4.0,17000,0,False,2024-07-21,2024-08-21,Concluído,2024,7,2024-07,31,17000
4,2026-09-03 15:31:26,Projeto Odisseia,Desktop,Marcos Vinícius,Delta Consult,4.0,7000,0,False,2024-08-01,2024-09-01,Concluído,2024,8,2024-08,31,7000


In [9]:
def construir_serie_mensal(df, coluna_data='data_inicio'):
    df = df.copy()
    df['ano_mes'] = df[coluna_data].dt.to_period('M')
    serie = df.groupby('ano_mes').agg(
        faturamento_total=('valor_total_projeto', 'sum'),
        faturamento_colaborativo=('valor_faturamento_colaborativo', 'sum'),
        faturamento_proprio=('valor_faturamento_proprio', 'sum'),
    )

    idx_completo = pd.period_range(serie.index.min(), serie.index.max(), freq='M')
    n_preenchidos = len(idx_completo) - len(serie)
    if n_preenchidos > 0:
        print(f'[info] {n_preenchidos} mes(es) sem projetos iniciados foram preenchidos com faturamento = 0.')
    serie = serie.reindex(idx_completo, fill_value=0)
    serie.index.name = 'ano_mes'
    serie = serie.reset_index()
    serie['ano_mes'] = serie['ano_mes'].astype(str)
    serie['indice_tempo'] = range(len(serie))
    serie['faturamento_total_acumulado'] = serie['faturamento_total'].cumsum()
    serie['faturamento_colaborativo_acumulado'] = serie['faturamento_colaborativo'].cumsum()
    return serie

serie = construir_serie_mensal(df)
serie

[info] 14 mes(es) sem projetos iniciados foram preenchidos com faturamento = 0.


,ano_mes,faturamento_total,faturamento_colaborativo,faturamento_proprio,indice_tempo,faturamento_total_acumulado,faturamento_colaborativo_acumulado
0,2024-03,2500,0,2500,0,2500,0
1,2024-04,5000,5000,0,1,7500,5000
2,2024-05,3000,0,3000,2,10500,5000
3,2024-06,0,0,0,3,10500,5000
4,2024-07,17000,0,17000,4,27500,5000
5,2024-08,27000,20000,7000,5,54500,25000
6,2024-09,10000,0,10000,6,64500,25000
7,2024-10,15000,0,15000,7,79500,25000
8,2024-11,35000,35000,0,8,114500,60000
9,2024-12,0,0,0,9,114500,60000
